# Fit Rossby Wave Components: Testing Windows

Curated from the archived research notebook `1. Extract rossby wave data(testing).ipynb`. Read the repository
README and `docs/limitations.md` before execution. All original files and
execution outputs were preserved separately.

Full preprocessing requires input files and an `internal_waves` module that
were not included in the available collection. See `docs/reproduction.md`.


In [ ]:
from pathlib import Path
import os
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'src/project_paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter from this repository or one of its notebook directories.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from project_paths import NotebookPaths
paths = NotebookPaths(PROJECT_ROOT, output_group='preprocessing')
input_path, input_glob, output_path = paths.input_path, paths.input_glob, paths.output_path
ALLOW_TRAINING = False  # Explicitly enable before running model training cells.


In [ ]:
import xarray as xr
import numpy as np
import scipy
import cmocean as cmo
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from glob2 import glob
import dask.array
from swath_rossby_wave import inversion
from tqdm import tqdm
from swath_rossby_wave import skill_matrix, build_h_matrix2, build_hswath_matrix2, inversion, make_error_over_time
import matplotlib.patches as mpatches
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from numpy import linalg as LA
import h5py


In [ ]:
# define longitude/latitude domain, based on grid points in AVISO subsampled files
lonidx_west, lonidx_east  =  76, 112
latidx_south, latidx_north = 27, 67


In [ ]:
# Open the .mat file
with h5py.File(input_path('aviso_tot_MSLA_ccs.mat'), 'r') as f:
    # Access the datasets using key access
    dsave = f['dsave'][:]  # Shape (188, 108, 10016)
    tsave = f['tsave'][:]  # Shape (1, 10016)
    xsave = f['xsave'][:]  # Shape (1, 188)
    ysave = f['ysave'][:]  # Shape (1, 108)

    # Print the data or shape of the arrays
    print("dsave shape:", dsave.shape)
    print("tsave shape:", tsave.shape)
    print("xsave shape:", xsave.shape)
    print("ysave shape:", ysave.shape)


In [ ]:
dsave_transpose = np.transpose(dsave, (1, 0, 2))
SSHA = dsave_transpose[latidx_south:latidx_north, lonidx_west:lonidx_east, :]
T_time = tsave * 86400 # in seconds
T_time = T_time.flatten()
lon, lat = (360 - xsave[0, lonidx_west:lonidx_east]) * -1, ysave[0, latidx_south:latidx_north]
dlon = lon - lon.mean()
dlat = lat - lat.mean()
tsave_flat = tsave.flatten()
date_time_all = np.array([np.datetime64(int(atime - tsave_flat[0] + 8401), 'D') for atime in tsave_flat])


In [ ]:
ssha_time_mean = SSHA[:, :, : ].mean(axis = -1) # remove multi-year mean (climatology)
ssha_time_mean_expanded = ssha_time_mean[:, :, np.newaxis]
# remove mean from SSH data to produce anomaly over full analysis period
SSHA = SSHA - ssha_time_mean_expanded


In [ ]:
#  alternately could remove 80-day mean  SSHA[day0 + day0 + 30].mean(axis = -1)
#  this is not recommended
SSHA_masked = np.ma.masked_invalid(SSHA)
ssha_mask = np.ma.getmask(SSHA_masked)


In [ ]:
# set Rossby wave model parameters
Phi0 = lat.mean() # central latitude (φ0)
Omega = 7.27e-5 # Ω is the angular speed of the earth
Earth_radius = 6.371e6 / 1e5 # meters
Beta = 2 * Omega * np.cos(Phi0*np.pi/180.) / Earth_radius
f0 =  2 * Omega * np.sin(Phi0*np.pi/180.) #1.0313e-4


In [ ]:
# number of modes to use---code only tested for MModes = 1
MModes = 1


In [ ]:
# set zonal and meridional wavenumber increments and upper/lower bounds
# nominally assume a 10 x 10 degree domain, though we actually use a slightly rectangular domain
L_lat = 10 # domain latitude length degree
L_lon = 10 # domain lognitude length

domain_factor = 1.1 # the smaller, the less waves

l_interval = 2 * np.pi / (domain_factor * L_lat) # zonal wavemenumber interval
k_interval = 2 * np.pi / (domain_factor * L_lon) # longitutional wavemenumber interval

lambda_min = 1.2 # 100km = 1 degree minimum wavelength resolved , the smaller, the more waves

k_min = 0
k_max = 2 * np.pi / lambda_min
l_max = k_max
l_min = -1 * k_max

# set range of k and l
k_n_orig = np.arange(k_min, k_max, k_interval) # degree^-1
l_n_orig = np.arange(l_min, l_max, l_interval) # degree^-1
l_n = l_n_orig.reshape(len(l_n_orig), MModes) #* 0 # lon, zonal propagration
k_n = k_n_orig.reshape(len(k_n_orig), MModes) #* 0 # lat, meridonal propagration
# set size of wavenumber domain
M = k_n.size * l_n.size


In [ ]:
# Open the NetCDF file using h5py
with h5py.File(input_path('stratification_sample_ccs.nc'), 'r') as f:
    # Access the 'Psi' dataset
    Psi = f['Psi'][:]
    Psi = Psi[:]
Rm = 5e4  / 1e5 # 50 km to degree
wavespeed = Rm * f0  # deg / s strat_ds.C2[:MModes].data
Rm = np.array([Rm]) #unit: degree


In [ ]:
# define covariance matrix (R over P)
counter = 0
exp=-2
k0 = l_n.max() # flat at or below k0

kl, R_over_P = np.zeros(2 * M), np.zeros([2 * M, 2 * M])
p_diagonal = np.zeros([2 * M])
k_, l_ = np.zeros(len(k_n)*len(l_n)), np.zeros(len(k_n)*len(l_n))
R = 0.01 # noise = 1. cm
#R=0.
counter=0

for kk in k_n:
    for ll in l_n:
        k_[counter] , l_[counter]  = kk, ll
        kl[counter] =  np.sqrt(kk ** 2 + ll ** 2) # wavenumber
        p_diagonal[2 * counter] = (kl[counter]+k0) ** exp
        p_diagonal[2 * counter + 1] = (kl[counter]+k0) ** exp
        counter += 1

R_over_P = np.zeros([2 * M, 2 * M])
p_factor = .16/p_diagonal.sum() # variance of the model,  convert sum of variance from waven number to meter
np.fill_diagonal(R_over_P[:],  R / p_diagonal / p_factor)


In [ ]:
#days_of_prediction=[10,21,31,42,52,63,73,84,94]
day0 = 0
day1 = 200
MSLA0 = SSHA_masked[:, :, day0:day1] #AVISO input
ndays = 200
H_all, SSH_vector = build_h_matrix2(MSLA0, MModes, k_n, l_n, lon, lat, T_time[:], Psi, Rm, 0)


In [ ]:
# create a dummy masked matrix in order to build the full H matrix
ssha_clean = np.ma.masked_invalid(np.zeros([MSLA0.shape[0],MSLA0.shape[1],MSLA0.shape[2]]))
# define matrices for first 100 days and last 100 days
H_all_full_grid, SSH_vector_full = build_h_matrix2(ssha_clean, MModes, k_n, l_n, lon, lat, T_time[:], Psi, Rm, day0-day0)


In [ ]:
# set standard deviation of error parameters
alpha_std = np.arange(5e-4, 3.05e-2, 1e-3)
# identify the alpha_std value to use for this case study
iuse_alpha=12
# set start dates for months
day0_array = np.arange(42, 42 + 168 * 59, 168)


In [ ]:
import xarray as xr
# loop through all start dates and extract AVISO data and parameters for study region
# find fitted coefficients for Rossby wave model
# save clean fake data
for day0 in day0_array[:]:
    day1=day0+ndays
    time_range = (day1 - day0) # forecast time range

    MSLA0 = SSHA_masked[:, :, day0:day1] #AVISO input
    date_time = date_time_all[day0:]

    variance_explained_inverse  = np.zeros(time_range)
    SSHA_vector=np.ma.compressed(MSLA0.transpose([2,0,1]).flatten())

    # least-squares fit to find best coefficients for non-land data points
    amp, ssh_estimated = inversion(SSHA_vector, H_all, R_over_P)

    # first 100 days
    MSLA_fwrd=np.matmul(H_all_full_grid, amp)
    MSLA_fwrd=MSLA_fwrd.reshape([ndays,MSLA0.shape[0],MSLA0.shape[1]])
    MSLA_fwrd=MSLA_fwrd.transpose((1,2,0))
    # MSLA_fwrd = np.ma.masked_where(np.ma.getmask(MSLA0),MSLA_fwrd)

    residual = SSHA_masked[:,:,day0:day0+time_range] - MSLA_fwrd

    variance_explained_inverse = (np.mean(residual**2,axis=(0,1))) / (np.var(SSHA_masked[:,:,day0:day0+time_range],axis=(0,1)))
    ds_output = xr.Dataset(data_vars={'l_n' : l_n[:, 0],
                                  'k_n' : k_n[:, 0],
                                  'Amplitudes': amp,
                                  'variance_explained': (('time'), 1-variance_explained_inverse),
                                  'MSLA_forward' : (('YC', 'XC', 'time'), MSLA_fwrd[:, :, :time_range]),
                                  'Rm': Rm,
                                  'XC' : (('XC'), lon.data),
                                  'YC' : (('YC'), lat.data),
                                  'time': date_time[:time_range]},
                      attrs = dict(description=('Data sample of the selected waves, amplitudes, estimated SSH anomalies and residual, fit with '
                                                + str(day1 - day0) + '-day prior data.')))
    ds_output.to_netcdf(output_path('./new_testing_data_rossby_wave_estimate_' + str(date_time[0])[:10] +'_' + str(k_n.size * l_n.size) +'waves_swotdomain_'+ str(int((day1 - day0))) +'days.nc'), engine='h5netcdf')
